# Titanic Data Analysis and JSON Export

**Author:** Kimia Asgari  

In [1]:
## Step 1 — Project setup

import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path("data")
CSV_FILE = DATA_DIR / "titanic.csv"
JSON_FILE = DATA_DIR / "titanic_processed.json"
DATA_DIR.mkdir(exist_ok=True)

print("Project setup complete!")
print(f"Data directory : {DATA_DIR.resolve()}")
print(f"CSV file       : {CSV_FILE}")
print(f"JSON output    : {JSON_FILE}")


Project setup complete!
Data directory : /Users/kimia/Documents/Ironhack/Week1/lab2_Data-Manipulation-JSON-Handling/data
CSV file       : data/titanic.csv
JSON output    : data/titanic_processed.json


In [8]:
## Step 2 — Import and explore the data

df = pd.read_csv('/Users/kimia/Documents/Ironhack/Week1/lab2_Data-Manipulation-JSON-Handling/data/train.csv')
print(f"Dataset loaded successfully! Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print("\nFirst few rows:")
display(df.head())
print("Data types:")
print(df.dtypes)

Dataset loaded successfully! Shape: (891, 12)

Columns: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

First few rows:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Data types:
PassengerId      int64
Survived         int64
Pclass           int64
Name               str
Sex                str
Age            float64
SibSp            int64
Parch            int64
Ticket             str
Fare           float64
Cabin              str
Embarked           str
dtype: object


In [9]:
## Step 3 — Descriptive statistics (mean / median / std)

numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
statistics = {}
for col in numeric_columns:
    statistics[col] = {
        "mean":   float(df[col].mean()),
        "median": float(df[col].median()),
        "std":    float(df[col].std()),
        "min":    float(df[col].min()),
        "max":    float(df[col].max()),
    }
stats_df = pd.DataFrame(statistics).T[["mean", "median", "std", "min", "max"]]
display(stats_df.round(3))

,mean,median,std,min,max
PassengerId,446.000,446.000,257.354,1.00,891.000
Survived,0.384,0.000,0.487,0.00,1.000
Pclass,2.309,3.000,0.836,1.00,3.000
Age,29.699,28.000,14.526,0.42,80.000
SibSp,0.523,0.000,1.103,0.00,8.000
Parch,0.382,0.000,0.806,0.00,6.000
Fare,32.204,14.454,49.693,0.00,512.329


In [10]:
## Step 4 — Missing values analysis

missing = {}
for col in df.columns:
    count = int(df[col].isna().sum())
    missing[col] = {"count": count, "percent": round(count / len(df) * 100, 2)}
missing_df = pd.DataFrame(missing).T.sort_values("count", ascending=False)
display(missing_df)
print("Columns with missing data (most -> least):",
      missing_df[missing_df["count"] > 0].index.tolist())

,count,percent
Cabin,687.0,77.10
Age,177.0,19.87
Embarked,2.0,0.22
PassengerId,0.0,0.00
Survived,0.0,0.00
Pclass,0.0,0.00
Name,0.0,0.00
Sex,0.0,0.00
SibSp,0.0,0.00
Parch,0.0,0.00


Columns with missing data (most -> least): ['Cabin', 'Age', 'Embarked']


In [17]:
## Step 5 — Feature engineering

def categorize_age(age):
    if pd.isna(age):      return "Unknown"
    elif age < 18:        return "Child"
    elif age < 30:        return "Young Adult"
    elif age < 50:        return "Adult"
    else:                 return "Senior"

def extract_title(name):
    if pd.isna(name) or "," not in name or "." not in name:
        return "Unknown"
    title = name.split(",")[1].split(".")[0].strip()
    if title.lower().startswith("the "):
        title = title[4:].strip()
    mapping = {"Mlle": "Miss", "Ms": "Miss", "Mme": "Mrs",
               "Lady": "Rare", "Countess": "Rare", "Capt": "Rare", "Col": "Rare",
               "Don": "Rare", "Dr": "Rare", "Major": "Rare", "Rev": "Rare",
               "Sir": "Rare", "Jonkheer": "Rare", "Dona": "Rare"}
    return mapping.get(title, title)


df_engineered = df.copy()
df_engineered["FamilySize"] = df_engineered["SibSp"] + df_engineered["Parch"] + 1
df_engineered["IsAlone"]    = (df_engineered["FamilySize"] == 1).astype(int)
df_engineered["AgeGroup"]   = df_engineered["Age"].apply(categorize_age)
df_engineered["Title"]      = df_engineered["Name"].apply(extract_title)

display(df_engineered[["Name","SibSp","Parch","FamilySize","IsAlone","Age","AgeGroup","Title"]].head(10))

print(df_engineered.groupby('IsAlone')['Survived'].mean().round(3))
print(df_engineered.groupby('AgeGroup')['Survived'].mean().round(3))
print(df_engineered.groupby('Title')['Survived'].mean().round(3).sort_values(ascending=False))

,Name,SibSp,Parch,FamilySize,IsAlone,Age,AgeGroup,Title
0,"Braund, Mr. Owen Harris",1,0,2,0,22.0,Young Adult,Mr
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,0,2,0,38.0,Adult,Mrs
2,"Heikkinen, Miss. Laina",0,0,1,1,26.0,Young Adult,Miss
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,0,2,0,35.0,Adult,Mrs
4,"Allen, Mr. William Henry",0,0,1,1,35.0,Adult,Mr
5,"Moran, Mr. James",0,0,1,1,NaN,Unknown,Mr
6,"McCarthy, Mr. Timothy J",0,0,1,1,54.0,Senior,Mr
7,"Palsson, Master. Gosta Leonard",3,1,5,0,2.0,Child,Master
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",0,2,3,0,27.0,Young Adult,Mrs
9,"Nasser, Mrs. Nicholas (Adele Achem)",1,0,2,0,14.0,Child,Mrs


IsAlone
0    0.506
1    0.304
Name: Survived, dtype: float64
AgeGroup
Adult          0.418
Child          0.540
Senior         0.365
Unknown        0.294
Young Adult    0.351
Name: Survived, dtype: float64
Title
Mrs       0.794
Miss      0.703
Master    0.575
Rare      0.348
Mr        0.157
Name: Survived, dtype: float64


In [22]:
## Feature analysis — survivors vs non-survivors

print("Family size by survival group:")
display(df_engineered.groupby("Survived")["FamilySize"].agg(["mean", "median", "std"]).round(3))

print("Survival rate by IsAlone (0 = with family, 1 = alone):")
display(df_engineered.groupby("IsAlone")["Survived"].mean().round(3))

print("Survival rate by AgeGroup:")
display(df_engineered.groupby("AgeGroup")["Survived"].mean().round(3))

print("Survival rate by Title:")
display(df_engineered.groupby("Title")["Survived"].mean().round(3).sort_values(ascending=False))

Family size by survival group:


,mean,median,std
Survived,,,
0,1.883,1.0,1.831
1,1.939,2.0,1.186


Survival rate by IsAlone (0 = with family, 1 = alone):


IsAlone
0    0.506
1    0.304
Name: Survived, dtype: float64

Survival rate by AgeGroup:


AgeGroup
Adult          0.418
Child          0.540
Senior         0.365
Unknown        0.294
Young Adult    0.351
Name: Survived, dtype: float64

Survival rate by Title:


Title
Mrs       0.794
Miss      0.703
Master    0.575
Rare      0.348
Mr        0.157
Name: Survived, dtype: float64

In [23]:
## Step 6 — Classes for structured JSON export

class Passenger:
    """Represents a single passenger and their engineered attributes."""
    def __init__(self, passenger_id, name, age, sex, survived, pclass, fare,
                 embarked=None, family_size=None, is_alone=None,
                 age_group=None, title=None):
        self.passenger_id = int(passenger_id) if pd.notna(passenger_id) else None
        self.name        = str(name)   if pd.notna(name) else None
        self.age         = float(age)  if pd.notna(age) else None
        self.sex         = str(sex)    if pd.notna(sex) else None
        self.survived    = int(survived) if pd.notna(survived) else None
        self.pclass      = int(pclass) if pd.notna(pclass) else None
        self.fare        = float(fare) if pd.notna(fare) else None
        self.embarked    = str(embarked) if pd.notna(embarked) else None
        self.family_size = int(family_size) if pd.notna(family_size) else None
        self.is_alone    = int(is_alone) if pd.notna(is_alone) else None
        self.age_group   = str(age_group) if pd.notna(age_group) else None
        self.title       = str(title) if pd.notna(title) else None

    def to_dict(self):
        return {
            "passenger_id": self.passenger_id, "name": self.name, "age": self.age,
            "sex": self.sex, "survived": self.survived, "pclass": self.pclass,
            "fare": self.fare, "embarked": self.embarked,
            "family_size": self.family_size, "is_alone": self.is_alone,
            "age_group": self.age_group, "title": self.title,
        }


class TitanicDataset:
    """Wraps the dataset: builds Passenger objects and exports to JSON."""
    def __init__(self, dataframe, statistics=None, missing=None):
        self.dataframe = dataframe
        self.statistics = statistics or {}
        self.missing = missing or {}
        self.passengers = []
        self._create_passengers()

    def _create_passengers(self):
        for _, row in self.dataframe.iterrows():
            self.passengers.append(Passenger(
                passenger_id=row.get("PassengerId"), name=row.get("Name"),
                age=row.get("Age"), sex=row.get("Sex"), survived=row.get("Survived"),
                pclass=row.get("Pclass"), fare=row.get("Fare"),
                embarked=row.get("Embarked"), family_size=row.get("FamilySize"),
                is_alone=row.get("IsAlone"), age_group=row.get("AgeGroup"),
                title=row.get("Title")))

    def get_summary_stats(self):
        total = len(self.passengers)
        survived = sum(1 for p in self.passengers if p.survived == 1)
        ages  = [p.age for p in self.passengers if p.age is not None]
        fares = [p.fare for p in self.passengers if p.fare is not None]
        return {
            "total_passengers": total, "survived": survived,
            "did_not_survive": total - survived,
            "survival_rate": round(survived / total, 4) if total else None,
            "average_age":  round(sum(ages) / len(ages), 2) if ages else None,
            "average_fare": round(sum(fares) / len(fares), 2) if fares else None,
        }

    def to_json(self, filename=JSON_FILE):
        data = {
            "metadata": {
                "dataset_name": "Titanic Passenger Dataset",
                "export_date": datetime.now().isoformat(),
                "total_passengers": len(self.passengers),
                "survival_rate": round(float(self.dataframe["Survived"].mean()), 4),
                "columns": list(self.dataframe.columns),
            },
            "statistics": self.statistics,
            "missing_values": self.missing,
            "summary": self.get_summary_stats(),
            "passengers": [p.to_dict() for p in self.passengers],
        }
        with open(filename, "w", encoding="utf-8") as f:
            json.dump(data, f, indent=2)
        print(f"Data exported to {filename}")
        return data


dataset = TitanicDataset(df_engineered, statistics=statistics, missing=missing)
print(f"Built {len(dataset.passengers)} Passenger objects.\n")
print("Summary statistics:")
for k, v in dataset.get_summary_stats().items():
    print(f"  {k}: {v}")
exported = dataset.to_json(JSON_FILE)

Built 891 Passenger objects.

Summary statistics:
  total_passengers: 891
  survived: 342
  did_not_survive: 549
  survival_rate: 0.3838
  average_age: 29.7
  average_fare: 32.2
Data exported to data/titanic_processed.json


In [24]:
## Step 7 — Testing and validation

with open(JSON_FILE, "r", encoding="utf-8") as f:
    json_data = json.load(f)

checks = [
    ("Top-level keys present",
     all(k in json_data for k in ["metadata","statistics","missing_values","summary","passengers"])),
    ("Passenger count matches DataFrame", len(json_data["passengers"]) == len(df_engineered)),
    ("Every passenger has 12 fields", all(len(p) == 12 for p in json_data["passengers"])),
    ("Summary total == passenger count",
     json_data["summary"]["total_passengers"] == len(json_data["passengers"])),
    ("Survived + did_not_survive == total",
     json_data["summary"]["survived"] + json_data["summary"]["did_not_survive"]
     == json_data["summary"]["total_passengers"]),
]
for label, ok in checks:
    print(f"  [{'PASS' if ok else 'FAIL'}] {label}")

print("\nSample passenger record:")
print(json.dumps(json_data["passengers"][0], indent=2))
print("\nSummary block:")
print(json.dumps(json_data["summary"], indent=2))
print("\nValidation", "PASSED" if all(ok for _, ok in checks) else "FAILED")

  [PASS] Top-level keys present
  [PASS] Passenger count matches DataFrame
  [PASS] Every passenger has 12 fields
  [PASS] Summary total == passenger count
  [PASS] Survived + did_not_survive == total

Sample passenger record:
{
  "passenger_id": 1,
  "name": "Braund, Mr. Owen Harris",
  "age": 22.0,
  "sex": "male",
  "survived": 0,
  "pclass": 3,
  "fare": 7.25,
  "embarked": "S",
  "family_size": 2,
  "is_alone": 0,
  "age_group": "Young Adult",
  "title": "Mr"
}

Summary block:
{
  "total_passengers": 891,
  "survived": 342,
  "did_not_survive": 549,
  "survival_rate": 0.3838,
  "average_age": 29.7,
  "average_fare": 32.2
}

Validation PASSED


## Summary write-up

**Engineered features:** `FamilySize` (SibSp + Parch + 1), `IsAlone`, `AgeGroup` (Child / Young Adult / Adult / Senior / Unknown), and `Title` (extracted from Name, with rare titles grouped).

**How they differentiate survival:** passengers travelling with family survived at 50.6% vs 30.4% for those travelling alone. Children survived at 54.0%, the highest of any age group, while Young Adults were lowest at 35.1%. `Title` is the strongest single signal: Mrs 79.4% and Miss 70.3% versus Mr 15.7% — effectively encoding the "women and children first" evacuation pattern.

**On using classes:** separating `Passenger` (row-to-object mapping and serialization) from `TitanicDataset` (aggregation and JSON export) kept the code organized and easy to extend — the JSON schema is defined once in `to_dict()`, so adding a field means editing one place rather than hunting through scattered dictionaries.